# GridOps-SoD: first experiment with LLM agents

Runs the experiment of `experiments/gridops` on Kaggle's free GPU.

**Before starting:** in the right panel, *Session options*, set **Accelerator = GPU T4 x2** and **Internet = On**.
Then *Run all*. The pilot takes a few minutes; the full run about 1–2 hours per model.
At the end, download `gridops_results.zip` from the *Output* panel (`/kaggle/working`).

In [ ]:
# 1. Code and dependencies
!git clone -q https://github.com/valeriobellandi/agentsod.git /kaggle/working/agentsod
%cd /kaggle/working/agentsod
!pip install -q pandapower
!apt-get -qq install -y zstd > /dev/null

In [ ]:
# 2. Ollama (OpenAI-compatible server on localhost:11434)
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags"); break
    except Exception:
        time.sleep(2)
print("ollama up")

In [ ]:
# 3. Models (open-weight, two families)
MODELS = ["qwen2.5:7b", "llama3.1:8b"]
for m in MODELS:
    !ollama pull {m}

In [ ]:
# 4. Pilot: 3 episodes per kind with the first model, to check that everything works
!python -m experiments.gridops.run --model {MODELS[0]} --n 3 --workers 4 --out results/gridops/pilot.jsonl
!python -m experiments.gridops.analyze results/gridops/pilot.jsonl | cut -d, -f1-8

In [ ]:
# 5. Full run: 40 episodes per kind (200 episodes x 3 conditions) per model.
#    Resumable: if the session stops, run this cell again and it continues where it left off.
N = 40
for m in MODELS:
    out = f"results/gridops/{m.replace(':', '-')}.jsonl"
    !python -m experiments.gridops.run --model {m} --n {N} --workers 4 --out {out}

In [ ]:
# 6. Summary and LaTeX table
!python -m experiments.gridops.analyze results/gridops/qwen2.5-7b.jsonl results/gridops/llama3.1-8b.jsonl \
    --csv results/gridops/summary.csv --tex results/gridops/table.tex | cut -d, -f1-8

In [ ]:
# 7. Package the results for download
!cp /kaggle/working/ollama.log /kaggle/working/agentsod/results/gridops/ 2>/dev/null
!cd /kaggle/working/agentsod && zip -qr /kaggle/working/gridops_results.zip results/gridops
!ls -la /kaggle/working/gridops_results.zip